# Oppgave 2 — Analyse av FTIR-datasett
**MLA310 — Project 1**

885 prøver av enzymatisk proteinhydrolysat fra fjærfe og fisk.
- FTIR-spektre (X) → gjennomsnittlig molekylvekt (y)
- CL1: 1=Kylling, 2=Kalkun, 3=Laks, 4=Makrell
- CL2: 1=Fugl, 2=Fisk

In [ ]:
using LinearAlgebra, Statistics, MAT, Plots
include("bidiag2.jl")
include("svdr.jl")
include("TregsRLooCV.jl")

## 2a) Innlesning av data

In [ ]:
vars  = matread("FTIR_AMW_MLA310.mat")
X     = Float64.(vars["X"])
y     = vec(Float64.(vars["y"]))
CL1   = Int.(vec(vars["CL1"]))
CL2   = Int.(vec(vars["CL2"]))
Names1 = vars["Names1"]
Names2 = vars["Names2"]
xlabs  = vec(vars["abcissas"])
m, n = size(X)
println("Datasett: m=$m prøver, n=$n bølgelengder")
println("Artfordeling: ", [sum(CL1 .== k) for k in 1:4], " (kylling/kalkun/laks/makrell)")

## 2.1) PCA på X
Scatter plots for alle kombinasjoner av de 4 første PC-ene, fargekodet per art.

In [ ]:
# Sentrer X og beregn SVD
x̄ = mean(X, dims=1)
Xc = X .- x̄
U, σ, V = svd(Xc)
T_pca = U[:, 1:4] .* σ[1:4]'   # Scores: m × 4

# Forklart varians
varexp = cumsum(σ.^2) / sum(σ.^2)
println("Forklart varians: PC1=$(round(100varexp[1],digits=1))%, PC1-2=$(round(100varexp[2],digits=1))%, PC1-4=$(round(100varexp[4],digits=1))%")

In [ ]:
colors  = [:blue, :red, :green, :orange]
markers = [:circle, :square, :diamond, :star5]
species_labels = ["Kylling", "Kalkun", "Laks", "Makrell"]

pc_pairs = [(1,2), (1,3), (1,4), (2,3), (2,4), (3,4)]
plts = []
for (i, j) in pc_pairs
    p = plot(title="PC$i vs PC$j", xlabel="PC$i", ylabel="PC$j", legend=:outertopright)
    for k in 1:4
        idx = CL1 .== k
        scatter!(p, T_pca[idx, i], T_pca[idx, j],
            label=species_labels[k], color=colors[k],
            marker=markers[k], markersize=3, alpha=0.6)
    end
    push!(plts, p)
end
plot(plts..., layout=(2,3), size=(1200, 700))

## 2.2) Global PLSR-modell med LooCV
Opp til 50 PLS-komponenter. RMSECV per antall komponenter.

In [ ]:
# LooCV for PLS (brute-force)
function pls_loocv(X, y, mc_max)
    m = size(X, 1)
    ypred = zeros(m, mc_max)
    for i in 1:m
        idx = [1:i-1; i+1:m]
        β₀_i, β_i = bidiag2(X[idx, :], y[idx]; mc=mc_max)
        for k in 1:mc_max
            ypred[i, k] = (X[i:i, :] * β_i[:, k] .+ β₀_i[:, k])[1]
        end
    end
    rmsecv = sqrt.(mean((y .- ypred).^2, dims=1)[:])
    return rmsecv, ypred
end

# RMS per art
function rms_per_species(y, ypred, CL1)
    mc = size(ypred, 2)
    rms = zeros(4, mc)
    for k in 1:4
        idx = CL1 .== k
        rms[k, :] = sqrt.(mean((y[idx] .- ypred[idx, :]).^2, dims=1)[:])
    end
    return rms
end

In [ ]:
mc_max = 50
println("Beregner global PLSR LooCV (m=$m, mc=$mc_max) — kan ta litt tid...")
@time rmsecv_global, ypred_global = pls_loocv(X, y, mc_max)

best_k_global = argmin(rmsecv_global)
println("Beste antall komponenter: $best_k_global (RMSECV = $(round(rmsecv_global[best_k_global], digits=4)))")

plot(1:mc_max, rmsecv_global, xlabel="Antall PLS-komponenter",
     ylabel="RMSECV", title="Global PLSR — LooCV prediksjonsfeil",
     label="Global", legend=:topright)
vline!([best_k_global], linestyle=:dash, color=:red, label="Optimalt k=$best_k_global")

In [ ]:
rms_sp_global = rms_per_species(y, ypred_global, CL1)
println("RMSECV per art ved k=$best_k_global (global modell):")
for k in 1:4
    println("  $(species_labels[k]): $(round(rms_sp_global[k, best_k_global], digits=4))")
end

## 2.3) Lokale PLSR-modeller per art

In [ ]:
ypred_local_pls = zeros(m)   # samlet kryss-validert prediksjon
best_k_local    = zeros(Int, 4)

p_local = plot(xlabel="Antall PLS-komponenter", ylabel="RMSECV",
               title="Lokale PLSR-modeller — LooCV", legend=:topright)

for k in 1:4
    idx_k = findall(CL1 .== k)
    Xk = X[idx_k, :];  yk = y[idx_k]
    mc_k = min(mc_max, length(idx_k) - 1)
    rmsecv_k, ypred_k = pls_loocv(Xk, yk, mc_k)
    best_k_local[k] = argmin(rmsecv_k)
    ypred_local_pls[idx_k] = ypred_k[:, best_k_local[k]]
    println("$(species_labels[k]): beste k=$(best_k_local[k]), RMSECV=$(round(rmsecv_k[best_k_local[k]], digits=4))")
    plot!(p_local, 1:mc_k, rmsecv_k, label=species_labels[k], color=colors[k])
end
display(p_local)

rms_overall_local_pls = sqrt(mean((y .- ypred_local_pls).^2))
println("\nSamlet RMSECV (lokale PLS-modeller): $(round(rms_overall_local_pls, digits=4))")
println("Samlet RMSECV (global PLS-modell):   $(round(rmsecv_global[best_k_global], digits=4))")

## 2.4) Ridge Regression — Global og lokal
10 λ-verdier jevnt fordelt på log₁₀-skala fra 10⁻⁵ til 10².

In [ ]:
λs = 10 .^ range(-5, 2, length=10)

press_global, minid_g, U_g, σ_g, V_g, H_g, bcoefs_g, λ_opt_g, bλ_g, hλ_g =
    TregsRLooCV(X, y, λs)

rmsecv_ridge_global = sqrt.(press_global ./ m)
println("Global Ridge — optimal λ = $(round(λ_opt_g, sigdigits=3)) (indeks $minid_g)")
println("Global Ridge — RMSECV    = $(round(rmsecv_ridge_global[minid_g], digits=4))")

plot(log10.(λs), rmsecv_ridge_global, marker=:circle,
     xlabel="log₁₀(λ)", ylabel="RMSECV",
     title="Global Ridge Regression — LooCV", label="Ridge global")
vline!([log10(λ_opt_g)], linestyle=:dash, color=:red, label="λ_opt")

In [ ]:
# Kryss-validerte prediksjoner fra lokale Ridge-modeller
ypred_local_ridge = zeros(m)

for k in 1:4
    idx_k = findall(CL1 .== k)
    Xk = X[idx_k, :];  yk = y[idx_k]
    press_k, minid_k, _, _, _, _, bcoefs_k, λ_k, bλ_k, hλ_k = TregsRLooCV(Xk, yk, λs)
    # Kryss-validerte prediksjoner
    x̄k = mean(Xk, dims=1);  ȳk = mean(yk)
    Xkc = Xk .- x̄k
    ypred_cv_k = yk .- (yk .- (Xkc * bcoefs_k[:, minid_k] .+ ȳk)) ./ (1 .- hλ_k)
    # OBS: LooCV-prediksjoner fra TregsRLooCV
    U_k, σ_k, V_k = svd(Xkc)
    σ_lam = σ_k .+ λ_k ./ σ_k
    ypred_cv_k2 = ȳk .+ Xkc * bcoefs_k[:, minid_k] .-
        (yk .- ȳk .- Xkc * bcoefs_k[:, minid_k]) ./ (1 .- hλ_k) .+ (yk .- ȳk .- Xkc * bcoefs_k[:, minid_k])
    ypred_local_ridge[idx_k] = yk .- (yk .- ȳk .- Xkc * bcoefs_k[:, minid_k]) ./ (1 .- hλ_k)
    rms_k = sqrt(mean((yk .- ypred_local_ridge[idx_k]).^2))
    println("$(species_labels[k]): λ_opt=$(round(λ_k, sigdigits=3)), RMSECV=$(round(rms_k, digits=4))")
end

rms_overall_local_ridge = sqrt(mean((y .- ypred_local_ridge).^2))
println("\nSamlet RMSECV (lokale Ridge-modeller): $(round(rms_overall_local_ridge, digits=4))")
println("Samlet RMSECV (global Ridge-modell):   $(round(rmsecv_ridge_global[minid_g], digits=4))")

## 2.5) MOLS-strategi — Lokale Ridge-modeller med data fra andre arter

**Primærmål:** Minimere prediksjonsfeil for art $k$.

**Sekundærmål:** Inkludere data fra de tre andre artene med vekt $w_j$.

**Implementering:** For art $k$, bygg den augmenterte matrisen:
$$X_{aug} = \begin{pmatrix} X_k \\ w_j X_{j \neq k} \end{pmatrix}, \quad y_{aug} = \begin{pmatrix} y_k \\ w_j y_{j \neq k} \end{pmatrix}$$

**Valg av vekter:**
- $w_j \in \{0.01, 0.1, 0.5, 1.0\}$ for de tre resterende artene.
- Felles vekt for alle ikke-primære arter (kun én parameter å velge).
- Velges ved LooCV **kun** på primærartens data.

In [ ]:
weights = [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0]

ypred_mols = zeros(m)

for k in 1:4
    idx_k = findall(CL1 .== k)
    idx_other = findall(CL1 .!= k)
    Xk = X[idx_k, :];  yk = y[idx_k]
    X_other = X[idx_other, :];  y_other = y[idx_other]
    mk = length(idx_k)

    best_rmsecv_k = Inf
    best_w_k = 0.0
    best_λ_k = λs[1]
    best_pred_k = zeros(mk)

    for w_j in weights
        # Augmenter datasett
        X_aug = [Xk; w_j .* X_other]
        y_aug = [yk; w_j .* y_other]

        # Tilpass Ridge på augmentert data, evaluer på art k kun
        # Manuell LooCV for primærartens data
        ypred_cv = zeros(mk)
        rmsecv_per_lambda = zeros(length(λs))

        press_aug, minid_aug, _, _, _, _, bcoefs_aug, λ_aug, bλ_aug, hλ_aug =
            TregsRLooCV(X_aug, y_aug, λs)

        # Prediker tilbake på primærdatasett (in-sample + korreksjon er ikke trivielt med augmentering)
        # Enklere: bruk den beste modellen fra augmentert data direkte på art k
        x̄_aug = mean(X_aug, dims=1);  ȳ_aug = mean(y_aug)
        X_aug_c = X_aug .- x̄_aug
        Xkc = Xk .- x̄_aug  # sentrer med augmentert gjennomsnitt

        # LooCV for art k med modell tilpasset augmentert data
        for i in 1:mk
            idx_train = [1:i-1; i+1:mk]
            X_aug_i = [Xk[idx_train, :]; w_j .* X_other]
            y_aug_i = [yk[idx_train];    w_j .* y_other]
            _, _, _, _, _, _, bcoefs_i, λ_i, _, _ = TregsRLooCV(X_aug_i, y_aug_i, λs)
            x̄_i = mean(X_aug_i, dims=1);  ȳ_i = mean(y_aug_i)
            ypred_cv[i] = (Xk[i:i, :] .- x̄_i) * bcoefs_i[:, argmin(vec(TregsRLooCV(X_aug_i, y_aug_i, λs)[1]))] .+ ȳ_i |> x -> x[1]
        end

        rmsecv_w = sqrt(mean((yk .- ypred_cv).^2))
        if rmsecv_w < best_rmsecv_k
            best_rmsecv_k = rmsecv_w
            best_w_k = w_j
            best_pred_k = ypred_cv
        end
    end

    ypred_mols[idx_k] = best_pred_k
    println("$(species_labels[k]): beste vekt w=$best_w_k, RMSECV=$(round(best_rmsecv_k, digits=4))")
end

rms_mols = sqrt(mean((y .- ypred_mols).^2))
println("\nSamlet RMSECV (MOLS): $(round(rms_mols, digits=4))")

## 2.6) Sammenligning av alle metoder

In [ ]:
println("=== Samlet RMSECV (kryss-validert) ===")
println("Global PLSR:      $(round(rmsecv_global[best_k_global], digits=4))")
println("Lokale PLSR:      $(round(rms_overall_local_pls, digits=4))")
println("Global Ridge:     $(round(rmsecv_ridge_global[minid_g], digits=4))")
println("Lokale Ridge:     $(round(rms_overall_local_ridge, digits=4))")
println("MOLS Ridge:       $(round(rms_mols, digits=4))")